In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into cached DataFrames (will reuse variable names if present)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to convert timestamp column to timezone-aware datetime (UTC->convert to Asia/Shanghai)
def to_ts_series(df, col='timestamp'):
    return pd.to_datetime(df[col], unit='s', utc=True).dt.tz_convert(tz)

# Add tz-aware timestamp columns (cached)
metric_df['ts'] = to_ts_series(metric_df)
trace_df['ts'] = to_ts_series(trace_df)
log_df['ts'] = to_ts_series(log_df)
error_logs_df['ts'] = to_ts_series(error_logs_df)

# Helper to build top cmdb_id counts (top 20)
def top_cmdb_counts(df, topn=20):
    vc = df['cmdb_id'].value_counts().head(topn).reset_index()
    vc.columns = ['cmdb_id', 'count']
    return vc

# Summaries for each file
summary_metric = pd.DataFrame([{
    'filename': 'metric.csv',
    'total_rows': len(metric_df),
    'min_timestamp': metric_df['ts'].min(),
    'max_timestamp': metric_df['ts'].max(),
    'top_cmdb_count_rows': top_cmdb_counts(metric_df),        # small dataframe (<=20 rows)
    'unique_kpi_names': list(pd.Index(metric_df['kpi_name'].dropna().unique())[:50])  # capped at 50
}])

summary_trace = pd.DataFrame([{
    'filename': 'trace.csv',
    'total_rows': len(trace_df),
    'min_timestamp': trace_df['ts'].min(),
    'max_timestamp': trace_df['ts'].max(),
    'top_cmdb_count_rows': top_cmdb_counts(trace_df),
    'unique_trace_names': list(pd.Index(trace_df['trace_name'].dropna().unique())[:50])
}])

summary_log = pd.DataFrame([{
    'filename': 'log.csv',
    'total_rows': len(log_df),
    'min_timestamp': log_df['ts'].min(),
    'max_timestamp': log_df['ts'].max(),
    'top_cmdb_count_rows': top_cmdb_counts(log_df),
    'unique_log_names': list(pd.Index(log_df['log_name'].dropna().unique())[:50])
}])

# error_logs: no KPI field, provide sample 3 distinct messages (or fewer if not available)
sample_messages = list(pd.Index(error_logs_df['message'].dropna().unique())[:3])
summary_error = pd.DataFrame([{
    'filename': 'error_logs.csv',
    'total_rows': len(error_logs_df),
    'min_timestamp': error_logs_df['ts'].min(),
    'max_timestamp': error_logs_df['ts'].max(),
    'top_cmdb_count_rows': top_cmdb_counts(error_logs_df),
    'sample_messages': sample_messages
}])

# For compactness, we will expose the summaries and the top_cmdb small tables and unique lists separately.
metric_top_cmdb = summary_metric.at[0, 'top_cmdb_count_rows']
metric_unique_kpis = summary_metric.at[0, 'unique_kpi_names']

trace_top_cmdb = summary_trace.at[0, 'top_cmdb_count_rows']
trace_unique_names = summary_trace.at[0, 'unique_trace_names']

log_top_cmdb = summary_log.at[0, 'top_cmdb_count_rows']
log_unique_names = summary_log.at[0, 'unique_log_names']

error_top_cmdb = summary_error.at[0, 'top_cmdb_count_rows']
error_sample_messages = summary_error.at[0, 'sample_messages']

# Display compact summaries and small top-cmdb tables + unique lists (variables only)
summary_metric, metric_top_cmdb, metric_unique_kpis, \
summary_trace, trace_top_cmdb, trace_unique_names, \
summary_log, log_top_cmdb, log_unique_names, \
summary_error, error_top_cmdb, error_sample_messages

```
Out[1]:
```
```python
# Summarize the previously loaded DataFrames succinctly as a single English string.
# This uses cached variables from the earlier execution: metric_df, trace_df, log_df, error_logs_df,
# and the compact helper variables metric_top_cmdb, metric_unique_kpis, trace_top_cmdb, trace_unique_names,
# log_top_cmdb, log_unique_names, error_top_cmdb, error_sample_messages.

def fmt_top(df, n=5):
    # df is a small DataFrame with columns ['cmdb_id','count']
    if df is None or df.empty:
        return "none"
    rows = df.head(n).itertuples(index=False)
    return ", ".join(f"{r.cmdb_id} ({r.count})" for r in rows)

summary = (
    "Telemetry load summary (times shown in UTC+8):\n\n"
    "1) metric.csv:\n"
    f"   - total rows: {len(metric_df)}\n"
    f"   - time range: {metric_df['ts'].min()} to {metric_df['ts'].max()}\n"
    f"   - top cmdb_id by row count (top 5): {fmt_top(metric_top_cmdb, 5)}\n"
    f"   - unique kpi_names: {len(metric_unique_kpis)} unique (showing up to 10 examples) -> "
    f"{', '.join(metric_unique_kpis[:10])}\n\n"
    "2) trace.csv:\n"
    f"   - total rows: {len(trace_df)}\n"
    f"   - time range: {trace_df['ts'].min()} to {trace_df['ts'].max()}\n"
    f"   - top cmdb_id by row count (top 5): {fmt_top(trace_top_cmdb, 5)}\n"
    f"   - unique trace_names: {len(trace_unique_names)} unique (showing up to 10 examples) -> "
    f"{', '.join(trace_unique_names[:10])}\n\n"
    "3) log.csv:\n"
    f"   - total rows: {len(log_df)}\n"
    f"   - time range: {log_df['ts'].min()} to {log_df['ts'].max()}\n"
    f"   - top cmdb_id by row count (top 5): {fmt_top(log_top_cmdb, 5)}\n"
    f"   - unique log_names: {len(log_unique_names)} unique -> {', '.join(log_unique_names)}\n\n"
    "4) error_logs.csv:\n"
    f"   - total rows: {len(error_logs_df)}\n"
    f"   - time range: {error_logs_df['ts'].min()} to {error_logs_df['ts'].max()}\n"
    f"   - top cmdb_id by row count (top 3): {fmt_top(error_top_cmdb, 3)}\n"
    f"   - sample distinct messages (up to 3): {', '.join(error_sample_messages)}\n\n"
    "Notes: Dataframes remain cached in the kernel (metric_df, trace_df, log_df, error_logs_df) for follow-up analysis."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     filename  total_rows             min_timestamp             max_timestamp                                top_cmdb_count_rows                                   unique_kpi_names
0  metric.csv      141612 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00                       cmdb_id  count
0         ...  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                 frontend-0   4680
3                 frontend-1   4560
4                frontend2-0   4560
5                 frontend-2   4140
6          checkoutservice-0   3540
7          checkoutservice-1   3540
8          checkoutservice-2   3300
9         checkoutservice2-0   3300
10   productcatalogservice-2   3240
11   productcatalogservice-0   3120
12   productcatalogservice-1   3120
13             cartservice-1   2760
14             cartservice-0   2640
15            cartservice2-0   2640
16              adservice2-0   2580
17               adservice-0   2580
18             cartservice-2   2520
19  productcatalogservice2-0   2520, ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'],     filename  total_rows             min_timestamp             max_timestamp                                top_cmdb_count_rows                                 unique_trace_names
0  trace.csv       29528 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00                      cmdb_id  count
0          ...  [trace.from_frontend-0.duration_mean, trace.fr...,                     cmdb_id  count
0                frontend-0   2576
1                frontend-2   2560
2                frontend-1   2312
3         checkoutservice-1   1588
4         checkoutservice-0   1560
5         checkoutservice-2   1560
6   productcatalogservice-0   1072
7               frontend2-0   1072
8   productcatalogservice-1   1068
9   productcatalogservice-2   1068
10       checkoutservice2-0   1044
11  recommendationservice-0    840
12  recommendationservice-1    840
13  recommendationservice-2    840
14            cartservice-0    716
15            cartservice-1    712
16            cartservice-2    704
17        currencyservice-2    636
18        currencyservice-1    636
19        currencyservice-0    624, ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-1.duration_mean', 'trace.to_cartservice-1.duration_p95', 'trace.to_cartservice-1.error_rate', 'trace.to_cartservice-1.row_count', 'trace.to_cartservice-2.duration_mean', 'trace.to_cartservice-2.duration_p95'],   filename  total_rows             min_timestamp             max_timestamp                                top_cmdb_count_rows                  unique_log_names
0  log.csv        1704 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00                      cmdb_id  count
0          ...  [log.error_count, log.row_count],                     cmdb_id  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
5             cartservice-2     60
6         currencyservice-1     60
7                frontend-0     60
8         currencyservice-0     60
9         currencyservice-2     60
10  productcatalogservice-2     60
11  productcatalogservice-0     60
12  productcatalogservice-1     60
13               frontend-2     60
14               frontend-1     60
15        shippingservice-1     60
16        shippingservice-2     60
17        shippingservice-0     60
18  recommendationservice-0     60
19  recommendationservice-1     60, ['log.error_count', 'log.row_count'],          filename  total_rows             min_timestamp             max_timestamp                                top_cmdb_count_rows                                    sample_messages
0  error_logs.csv         536 2022-03-20 08:30:02+08:00 2022-03-20 08:59:58+08:00        cmdb_id  count
0  frontend-0    229
1  f...  [severity: warning, message: failed to retriev...,       cmdb_id  count
0  frontend-0    229
1  frontend-2    227
2  frontend-1     80, ['severity: warning, message: failed to retrieve ads'])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse cached metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for every (cmdb_id, kpi_name) using the entire metric_df
thresholds_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric_df to the incident time window (inclusive)
start = pd.Timestamp('2022-03-20 08:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 08:59:00', tz=tz)
metric_window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()

# 3) Join thresholds and mark anomalies (value >= p95 OR value <= p5)
metric_window_df = metric_window_df.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
metric_window_df['is_anomaly'] = metric_window_df.apply(
    lambda r: pd.notna(r['p95']) and (r['value'] >= r['p95']) or pd.notna(r['p5']) and (r['value'] <= r['p5']),
    axis=1
)

# 4) Aggregate anomalies by (cmdb_id, kpi_name) and prepare top 20 by anomaly count
anomalies_df = metric_window_df[metric_window_df['is_anomaly']].copy()  # full anomaly rows cached

agg = (
    anomalies_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False, sort=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        earliest_anomaly_ts=('ts', 'min'),
        latest_anomaly_ts=('ts', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        p95=('p95', 'first'),
        p5=('p5', 'first')
    )
)

top20_agg_df = agg.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Keep outputs compact: show only the aggregated top-20 anomalies table
top20_agg_df

```
Out[2]:
```
```python
# Succinct plain-English summary of the anomaly analysis (uses cached anomalies_df and top20_agg_df)
summary = (
    "Anomaly analysis summary (window 2022-03-20 08:30:00 to 08:59:00, UTC+8):\n"
    "- Global P95/P5 thresholds were computed per (cmdb_id, kpi_name) using the entire metric series.\n"
    "- Windowed data (08:30–08:59) was marked as anomalous when value >= P95 or value <= P5.\n"
    "- Top results (top 20 by anomaly count): anomalies are concentrated in two groups:\n"
    "  1) shippingservice2-0: several container.node-5.* KPIs each show anomaly_count=30 spanning the entire window (08:30–08:59). Values are constant per KPI (e.g., 0, 20, 65536), equal to the computed thresholds.\n"
    "  2) adservice: many runtime.java_lang_GarbageCollector_LastGcInfo_* KPIs each show anomaly_count=30 across the full window, with constant extreme values (large integers, zeros, or -1) matching p95/p5.\n"
    "- Interpretation: these top-20 KPI pairs were anomalous at every timestamp in the 30-minute window (30 minutes × 1 reading/min = 30), indicating persistent extreme/constant readings for those specific metrics (container.node-5.* on shippingservice2-0 and GC-related fields on adservice).\n"
    "- Kernel variables saved for follow-up: thresholds_df (global p95/p5 per pair), metric_window_df (windowed data with thresholds and is_anomaly flag), anomalies_df (all anomaly rows in the window), top20_agg_df (the compact top-20 aggregate table).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts     min_value     max_value           p95            p5
0   shippingservice2-0  container.node-5.container_tasks_state.uninter...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1   shippingservice2-0                 container.node-5.container_threads             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  2.000000e+01  2.000000e+01  2.000000e+01  2.000000e+01
2   shippingservice2-0             container.node-5.container_threads_max             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3   shippingservice2-0  container.node-5.container_ulimits_soft.max_op...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  6.553600e+04  6.553600e+04  6.553600e+04  6.553600e+04
4   shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  1.048576e+08  1.048576e+08  1.048576e+08  1.048576e+08
6            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  6.935808e+06  6.935808e+06  6.935808e+06  6.935808e+06
7            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  3.493760e+06  3.493760e+06  3.493760e+06  3.493760e+06
8            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  3.407104e+06  3.407104e+06  3.407104e+06  3.407104e+06
9            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
10           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
11           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  3.004526e+07  3.004526e+07  3.004526e+07  3.004526e+07
12           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  1.073742e+09  1.073742e+09  1.073742e+09  1.073742e+09
13           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  4.194304e+07  4.194304e+07  4.194304e+07  4.194304e+07
14           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  4.194304e+07  4.194304e+07  4.194304e+07  4.194304e+07
15           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00 -1.000000e+00 -1.000000e+00 -1.000000e+00 -1.000000e+00
16           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00 -1.000000e+00 -1.000000e+00 -1.000000e+00 -1.000000e+00
17           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  5.242880e+06  5.242880e+06  5.242880e+06  5.242880e+06
18           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  5.242880e+06  5.242880e+06  5.242880e+06  5.242880e+06
19           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  1.048576e+08  1.048576e+08  1.048576e+08  1.048576e+08```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Aggregate anomalies_df by cmdb_id
anomalies_by_component = (
    anomalies_df
    .groupby('cmdb_id', sort=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        distinct_kpis=('kpi_name', pd.Series.nunique),
        earliest_anomaly_ts=('ts', 'min'),
        latest_anomaly_ts=('ts', 'max')
    )
    .reset_index()
    .sort_values('anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) Aggregate error_logs_df by minute in the incident window
start = pd.Timestamp('2022-03-20 08:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 08:59:59', tz=tz)

err_win = error_logs_df[(error_logs_df['ts'] >= start) & (error_logs_df['ts'] <= end)].copy()
err_win['minute_ts'] = err_win['ts'].dt.floor('min')

# total logs per minute
minute_counts = err_win.groupby('minute_ts', sort=False).size().rename('log_count').reset_index()

# top cmdb_id per minute (most logs that minute)
cmdb_counts = err_win.groupby(['minute_ts', 'cmdb_id'], sort=False).size().rename('cnt').reset_index()
top_cmdb = (
    cmdb_counts
    .sort_values(['minute_ts', 'cnt'], ascending=[True, False])
    .groupby('minute_ts', sort=False)
    .first()
    .reset_index()
    .rename(columns={'cmdb_id': 'top_cmdb_id', 'cnt': 'top_cmdb_count'})[['minute_ts', 'top_cmdb_id', 'top_cmdb_count']]
)

# top message sample per minute (most frequent message)
msg_counts = err_win.groupby(['minute_ts', 'message'], sort=False).size().rename('cnt').reset_index()
top_msg = (
    msg_counts
    .sort_values(['minute_ts', 'cnt'], ascending=[True, False])
    .groupby('minute_ts', sort=False)
    .first()
    .reset_index()
    .rename(columns={'message': 'top_message_sample', 'cnt': 'top_message_count'})[['minute_ts', 'top_message_sample', 'top_message_count']]
)

# merge into final per-minute summary and keep top 20 minutes by log_count
error_logs_per_min = (
    minute_counts
    .merge(top_cmdb, on='minute_ts', how='left')
    .merge(top_msg, on='minute_ts', how='left')
    .sort_values('log_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Display compact results (kernel variables)
anomalies_by_component, error_logs_per_min

```
Out[3]:
```
```python
# Plain-English summary of the two aggregates (uses cached anomalies_by_component and error_logs_per_min)
summary = (
    "Summary of findings (incident window 2022-03-20 08:30–08:59 UTC+8):\n\n"
    "1) Anomalies by component:\n"
    "- adservice2 (anomaly_count=9177, distinct_kpis=333) and adservice (anomaly_count=9109, distinct_kpis=341) are the top two components with the most metric anomalies.\n"
    "- Several frontend pods are also high on the list (frontend-1:2857, frontend-0:2829, frontend2-0:2823, frontend-2:2293).\n"
    "- Many other service pods (checkoutservice, productcatalogservice, cartservice, etc.) show substantial anomaly counts as well.\n"
    "- For these components anomalies cover the full window (earliest 08:30, latest 08:59), indicating persistent anomalous readings across the 30-minute incident period.\n\n"
    "2) Error logs per minute:\n"
    "- Peak minutes have 21 logs (examples: 08:33, 08:42, 08:48, 08:44), with top_cmdb_id usually a frontend pod (frontend-0 or frontend-2).\n"
    "- The most common log message in those minutes is a repeated warning: 'severity: warning, message: failed to retrieve...' (dominant in top minutes).\n"
    "- This pattern suggests frontends were repeatedly emitting the same warning across many minutes during the incident.\n\n"
    "Saved kernel variables for follow-up: anomalies_by_component, error_logs_per_min (both are available in memory for deeper inspection).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  anomaly_count  distinct_kpis       earliest_anomaly_ts         latest_anomaly_ts
0                adservice2           9177            333 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
1                 adservice           9109            341 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
2                frontend-1           2857            152 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
3                frontend-0           2829            156 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
4               frontend2-0           2823            152 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
5                frontend-2           2293            138 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
6         checkoutservice-0           2282            118 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
7         checkoutservice-1           2269            118 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
8        checkoutservice2-0           2169            110 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
9   productcatalogservice-2           2163            108 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
10  productcatalogservice-0           2067            104 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
11  productcatalogservice-1           2038            104 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
12             adservice2-0           2017             86 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
13              adservice-0           2010             86 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
14            cartservice-1           1992             92 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
15              adservice-1           1946             82 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
16            cartservice-0           1894             88 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
17           cartservice2-0           1875             88 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
18        checkoutservice-2           1850            110 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00
19              adservice-2           1815             78 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00,                    minute_ts  log_count top_cmdb_id  top_cmdb_count                                 top_message_sample  top_message_count
0  2022-03-20 08:33:00+08:00         21  frontend-0              10  severity: warning, message: failed to retrieve...                 21
1  2022-03-20 08:42:00+08:00         21  frontend-0               9  severity: warning, message: failed to retrieve...                 21
2  2022-03-20 08:48:00+08:00         21  frontend-2              11  severity: warning, message: failed to retrieve...                 21
3  2022-03-20 08:44:00+08:00         21  frontend-2              10  severity: warning, message: failed to retrieve...                 21
4  2022-03-20 08:38:00+08:00         20  frontend-0               9  severity: warning, message: failed to retrieve...                 20
5  2022-03-20 08:56:00+08:00         20  frontend-2              11  severity: warning, message: failed to retrieve...                 20
6  2022-03-20 08:54:00+08:00         20  frontend-2              12  severity: warning, message: failed to retrieve...                 20
7  2022-03-20 08:46:00+08:00         19  frontend-2               9  severity: warning, message: failed to retrieve...                 19
8  2022-03-20 08:59:00+08:00         19  frontend-0              10  severity: warning, message: failed to retrieve...                 19
9  2022-03-20 08:52:00+08:00         19  frontend-2               9  severity: warning, message: failed to retrieve...                 19
10 2022-03-20 08:31:00+08:00         19  frontend-2              10  severity: warning, message: failed to retrieve...                 19
11 2022-03-20 08:40:00+08:00         19  frontend-0               9  severity: warning, message: failed to retrieve...                 19
12 2022-03-20 08:36:00+08:00         18  frontend-2               8  severity: warning, message: failed to retrieve...                 18
13 2022-03-20 08:34:00+08:00         18  frontend-0               7  severity: warning, message: failed to retrieve...                 18
14 2022-03-20 08:57:00+08:00         18  frontend-0               8  severity: warning, message: failed to retrieve...                 18
15 2022-03-20 08:50:00+08:00         18  frontend-2               9  severity: warning, message: failed to retrieve...                 18
16 2022-03-20 08:49:00+08:00         18  frontend-0               8  severity: warning, message: failed to retrieve...                 18
17 2022-03-20 08:47:00+08:00         17  frontend-0               9  severity: warning, message: failed to retrieve...                 17
18 2022-03-20 08:32:00+08:00         17  frontend-0               8  severity: warning, message: failed to retrieve...                 17
19 2022-03-20 08:30:00+08:00         17  frontend-0               7  severity: warning, message: failed to retrieve...                 17)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse cached trace_df and tz
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for every (cmdb_id, trace_name) using the entire trace_df
traces_thresholds_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter trace_df to the incident time window (inclusive)
start = pd.Timestamp('2022-03-20 08:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 08:59:00', tz=tz)
trace_window_df = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].copy()

# 3) Join thresholds and mark anomalies (value >= p95 OR value <= p5)
trace_window_df = trace_window_df.merge(traces_thresholds_df, on=['cmdb_id', 'trace_name'], how='left')
trace_window_df['is_anomaly'] = (
    (trace_window_df['p95'].notna() & (trace_window_df['value'] >= trace_window_df['p95'])) |
    (trace_window_df['p5'].notna() & (trace_window_df['value'] <= trace_window_df['p5']))
)

# Save full anomaly rows
trace_anomalies_df = trace_window_df[trace_window_df['is_anomaly']].copy()

# 4) Aggregate anomalies by (cmdb_id, trace_name) and get top 20 by anomaly_count
trace_top20_agg_df = (
    trace_anomalies_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False, sort=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        earliest_anomaly_ts=('ts', 'min'),
        latest_anomaly_ts=('ts', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        p95=('p95', 'first'),
        p5=('p5', 'first')
    )
    .sort_values('anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5) Filter trace_top20_agg_df for trace_name containing 'adservice' (case-insensitive)
trace_top20_adservice_df = trace_top20_agg_df[trace_top20_agg_df['trace_name'].str.contains('adservice', case=False, na=False)].reset_index(drop=True)

# Display compact results (limited to 20 rows each)
traces_thresholds_df.head(20), trace_top20_agg_df, trace_top20_adservice_df

```
Out[4]:
```
```python
# Plain-English summary of the trace analysis (uses cached trace_top20_agg_df and trace_top20_adservice_df)
summary = (
    "Trace analysis summary (window 2022-03-20 08:30–08:59, UTC+8):\n"
    "- Global P95/P5 thresholds were computed per (cmdb_id, trace_name) using the full trace series.\n"
    "- Windowed traces were marked anomalous when value >= P95 or value <= P5.\n"
    "- Top-20 anomalous trace pairs are dominated by error_rate metrics (many trace.*.error_rate) across multiple services; each top row has anomaly_count=30 (i.e., anomalous at every minute in the 30-minute window).\n"
    "- Observed values for these top error_rate entries are 0.0 with p95 and p5 also 0.0, so they were flagged because they equal the computed threshold (persistent/constant readings across the window).\n"
    "- Although adservice entries appear in the thresholds table, filtering trace names for the substring 'adservice' returned no rows (trace_name values do not contain the substring 'adservice'), so no top-20 trace_name includes that substring.\n"
    "- Saved kernel variables for follow-up: traces_thresholds_df, trace_window_df, trace_anomalies_df, trace_top20_agg_df."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                           trace_name        p95        p5
0   adservice-0  trace.from_frontend-0.duration_mean   0.000016  0.000012
1   adservice-0   trace.from_frontend-0.duration_p95   0.000030  0.000013
2   adservice-0     trace.from_frontend-0.error_rate   0.000000  0.000000
3   adservice-0      trace.from_frontend-0.row_count   9.550000  6.000000
4   adservice-0  trace.from_frontend-1.duration_mean   0.000016  0.000011
5   adservice-0   trace.from_frontend-1.duration_p95   0.000019  0.000011
6   adservice-0     trace.from_frontend-1.error_rate   0.000000  0.000000
7   adservice-0      trace.from_frontend-1.row_count   4.000000  2.000000
8   adservice-0  trace.from_frontend-2.duration_mean   0.000016  0.000012
9   adservice-0   trace.from_frontend-2.duration_p95   0.000022  0.000013
10  adservice-0     trace.from_frontend-2.error_rate   0.000000  0.000000
11  adservice-0      trace.from_frontend-2.row_count  11.000000  5.000000
12  adservice-1  trace.from_frontend-0.duration_mean   0.000016  0.000012
13  adservice-1   trace.from_frontend-0.duration_p95   0.000024  0.000013
14  adservice-1     trace.from_frontend-0.error_rate   0.000000  0.000000
15  adservice-1      trace.from_frontend-0.row_count   9.550000  6.000000
16  adservice-1  trace.from_frontend-1.duration_mean   0.000015  0.000011
17  adservice-1   trace.from_frontend-1.duration_p95   0.000022  0.000012
18  adservice-1     trace.from_frontend-1.error_rate   0.000000  0.000000
19  adservice-1      trace.from_frontend-1.row_count   4.000000  1.000000,                     cmdb_id                                      trace_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  min_value  max_value  p95   p5
0               adservice-0                trace.from_frontend-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
1   productcatalogservice-2                trace.from_frontend-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
2   productcatalogservice-2                trace.from_frontend-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
3   productcatalogservice-2   trace.from_recommendationservice-0.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
4   productcatalogservice-2   trace.from_recommendationservice-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
5   productcatalogservice-1   trace.from_recommendationservice-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
6   productcatalogservice-1   trace.from_recommendationservice-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
7   productcatalogservice-1  trace.from_recommendationservice2-0.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
8   productcatalogservice-0                trace.from_frontend-0.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
9   productcatalogservice-2                trace.from_frontend-0.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
10  productcatalogservice-0                trace.from_frontend-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
11  productcatalogservice-0                trace.from_frontend-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
12  productcatalogservice-0   trace.from_recommendationservice-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
13  productcatalogservice-0   trace.from_recommendationservice-0.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
14               frontend-2     trace.to_recommendationservice-0.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
15               frontend-2     trace.to_recommendationservice-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
16               frontend-2     trace.to_productcatalogservice-1.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
17               frontend-2     trace.to_productcatalogservice-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
18               frontend-2     trace.to_recommendationservice-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0
19  productcatalogservice-0   trace.from_recommendationservice-2.error_rate             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00        0.0        0.0  0.0  0.0, Empty DataFrame
Columns: [cmdb_id, trace_name, anomaly_count, earliest_anomaly_ts, latest_anomaly_ts, min_value, max_value, p95, p5]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse cached anomalies_df from kernel
# Define target cmdb list and substrings
target_list = ['adservice','adservice-0','adservice-1','adservice-2','adservice2','adservice2-0']
substrs = ['GarbageCollector', 'HeapMemory', 'Memory', 'LastGcInfo']
pattern = '|'.join(substrs)

# Filter anomalies_df: cmdb_id in list OR startswith 'adservice' (case-sensitive)
cond_cmdb = anomalies_df['cmdb_id'].isin(target_list) | anomalies_df['cmdb_id'].str.startswith('adservice', na=False)
# kpi_name contains any of the substrings (case-sensitive)
cond_kpi = anomalies_df['kpi_name'].str.contains(pattern, na=False)

filtered = anomalies_df[cond_cmdb & cond_kpi].copy()

# Aggregate by cmdb_id and kpi_name
ad_gc_anomalies_df = (
    filtered
    .groupby(['cmdb_id', 'kpi_name'], as_index=False, sort=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        earliest_anomaly_ts=('ts', 'min'),
        latest_anomaly_ts=('ts', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        p95=('p95', 'first'),
        p5=('p5', 'first')
    )
    .sort_values('anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Display compact result (saved in kernel variable ad_gc_anomalies_df)
ad_gc_anomalies_df

```
Out[5]:
```
```python
# Compose a plain-English summary of the filtered adservice GC/Memory anomalies.
# Uses cached kernel variable: ad_gc_anomalies_df
df = ad_gc_anomalies_df

n_rows = len(df)
top_cmdb_counts = df['cmdb_id'].value_counts().head(5).to_dict()
unique_anomaly_counts = sorted(df['anomaly_count'].unique())
sample_kpis = df['kpi_name'].unique()[:6].tolist()

summary = (
    f"Adservice memory/GC anomaly summary (compact):\n"
    f"- Rows returned: {n_rows} (aggregated by cmdb_id & kpi_name).\n"
    f"- Top components in this set: {', '.join(f'{k} ({v})' for k,v in top_cmdb_counts.items())}.\n"
    f"- Anomaly counts observed (unique values): {unique_anomaly_counts} — most entries show anomaly_count=30 (i.e., anomalous at every minute in the 30-min window).\n"
    f"- KPI focus: memory and GC-related metrics (examples: {', '.join(sample_kpis)}).\n"
    f"- Observed pattern: values are constant per KPI (min == max) and equal the computed p95/p5 thresholds, indicating persistent extreme or fixed readings for these memory/GC KPIs on adservice instances (notably adservice2).\n"
    f"- Saved variable: ad_gc_anomalies_df (available in kernel for further inspection).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts    min_value    max_value          p95           p5
0   adservice2      runtime.java_nio_BufferPool_MemoryUsed.mapped             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          0.0          0.0          0.0          0.0
1   adservice2      runtime.java_nio_BufferPool_MemoryUsed.direct             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00      57344.0      57344.0      57344.0      57344.0
2   adservice2  runtime.java_lang_MemoryPool_Usage_committed.T...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00   20455424.0   20455424.0   20455424.0   20455424.0
3   adservice2  runtime.java_lang_MemoryPool_Usage_init.Code_C...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00    2555904.0    2555904.0    2555904.0    2555904.0
4   adservice2  runtime.java_lang_MemoryPool_Usage_init.Compre...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          0.0          0.0          0.0          0.0
5   adservice2  runtime.java_lang_MemoryPool_Usage_init.Eden_S...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00    2228224.0    2228224.0    2228224.0    2228224.0
6   adservice2  runtime.java_lang_MemoryPool_Usage_init.Metaspace             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          0.0          0.0          0.0          0.0
7   adservice2  runtime.java_lang_MemoryPool_Usage_init.Surviv...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00     262144.0     262144.0     262144.0     262144.0
8   adservice2  runtime.java_lang_MemoryPool_Usage_init.Tenure...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00    5636096.0    5636096.0    5636096.0    5636096.0
9   adservice2  runtime.java_lang_MemoryPool_Usage_max.Code_Cache             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00  251658240.0  251658240.0  251658240.0  251658240.0
10  adservice2  runtime.java_lang_MemoryPool_UsageThresholdSup...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          1.0          1.0          1.0          1.0
11  adservice2  runtime.java_lang_MemoryPool_UsageThresholdSup...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          0.0          0.0          0.0          0.0
12  adservice2  runtime.java_lang_MemoryPool_UsageThresholdSup...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          1.0          1.0          1.0          1.0
13  adservice2  runtime.java_lang_MemoryPool_Usage_committed.C...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00   22872064.0   22872064.0   22872064.0   22872064.0
14  adservice2  runtime.java_lang_MemoryPool_Usage_committed.C...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00    3801088.0    3801088.0    3801088.0    3801088.0
15  adservice2  runtime.java_lang_MemoryPool_Usage_committed.E...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00    8323072.0    8323072.0    8323072.0    8323072.0
16  adservice2  runtime.java_lang_MemoryPool_Usage_committed.M...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00   31326208.0   31326208.0   31326208.0   31326208.0
17  adservice2  runtime.java_lang_MemoryPool_Usage_committed.S...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00     983040.0     983040.0     983040.0     983040.0
18  adservice2  runtime.java_lang_MemoryPool_UsageThresholdCou...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          0.0          0.0          0.0          0.0
19  adservice2  runtime.java_lang_MemoryPool_UsageThresholdExc...             30 2022-03-20 08:30:00+08:00 2022-03-20 08:59:00+08:00          0.0          0.0          0.0          0.0```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse cached variables: metric_window_df, error_logs_per_min
tz = pytz.timezone('Asia/Shanghai')

# Select minutes with log_count >= 20
minutes_df = error_logs_per_min[error_logs_per_min['log_count'] >= 20].copy()

# Prepare anomalies-only from metric_window_df
anomalies_only = metric_window_df[metric_window_df['is_anomaly']].copy()

# Aggregate anomaly counts by exact minute timestamp and cmdb_id
anomalies_by_min = (
    anomalies_only
    .groupby(['ts', 'cmdb_id'], sort=False)
    .size()
    .rename('anomaly_count')
    .reset_index()
    .rename(columns={'ts': 'minute_ts'})
)

# Helper to format top up to 3 components for a minute
def fmt_top_components(df_min):
    if df_min is None or df_min.empty:
        return ""
    top = df_min.sort_values('anomaly_count', ascending=False).head(3)
    return ", ".join(f"{r.cmdb_id} ({r.anomaly_count})" for r in top.itertuples(index=False))

# Build per-minute breakdown (include minutes even if no anomalies)
rows = []
for _, mrow in minutes_df[['minute_ts', 'log_count']].iterrows():
    minute = mrow['minute_ts']
    log_count = int(mrow['log_count'])
    grp = anomalies_by_min[anomalies_by_min['minute_ts'] == minute]
    top_components = fmt_top_components(grp)
    rows.append({'minute_ts': minute, 'log_count': log_count, 'top_components': top_components})

minute_component_anomalies = pd.DataFrame(rows)

# Sort by minute_ts ascending for readability (could also sort by log_count)
minute_component_anomalies = minute_component_anomalies.sort_values('minute_ts').reset_index(drop=True)

# Display compact result (one row per selected minute, <=20 rows)
minute_component_anomalies

```
Out[6]:
```
```python
# Compact plain-English summary based on the cached minute_component_anomalies table
from collections import Counter

# use cached minute_component_anomalies
n_minutes = len(minute_component_anomalies)
minutes_list = minute_component_anomalies['minute_ts'].astype(str).tolist()

# extract top component names from the top_components strings (first 3 items per row)
comp_names = []
for s in minute_component_anomalies['top_components'].fillna(''):
    parts = [p.strip() for p in s.split(',') if p.strip()]
    for p in parts[:3]:
        name = p.split(' ')[0]
        comp_names.append(name)
top_comp_counts = Counter(comp_names).most_common(5)

summary = (
    f"Minutes with >=20 error logs: {n_minutes} minutes ({', '.join(minutes_list)}).\n"
    f"For these minutes, metric anomalies are dominated by ad-related components: "
    f"{', '.join(f'{c} ({cnt})' for c,cnt in top_comp_counts[:3])} (most frequent top contributors).\n"
    f"Typical per-minute anomaly counts for adservice/adservice2 are ~300 anomalies each at peak minutes, "
    f"with frontend pods often appearing next. This indicates a strong temporal correlation between frontend error-log spikes and heavy metric anomalies on adservice/adservice2 during the incident.\n"
    f"Saved kernel variable: minute_component_anomalies (per-minute component anomaly breakdown)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

minute_ts  log_count                                     top_components
0 2022-03-20 08:33:00+08:00         21  adservice (306), adservice2 (305), frontend-0 ...
1 2022-03-20 08:38:00+08:00         20  adservice2 (302), adservice (299), frontend-1 ...
2 2022-03-20 08:42:00+08:00         21  adservice2 (301), adservice (297), frontend-1 ...
3 2022-03-20 08:44:00+08:00         21  adservice2 (300), adservice (299), frontend-0 ...
4 2022-03-20 08:48:00+08:00         21  adservice (302), adservice2 (302), frontend-1 ...
5 2022-03-20 08:54:00+08:00         20  adservice2 (304), adservice (301), checkoutser...
6 2022-03-20 08:56:00+08:00         20  adservice2 (301), adservice (298), frontend-1 ...```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cached variables: error_logs_per_min, anomalies_df
# 1) pick top 2 minutes by log_count (tie -> earliest)
failure_minutes = (
    error_logs_per_min
    .sort_values(['log_count', 'minute_ts'], ascending=[False, True])
    .head(2)['minute_ts']
    .tolist()
)

# Components of interest
components = ['adservice','adservice-0','adservice-1','adservice-2',
              'adservice2','adservice2-0','frontend-0','frontend-1','frontend-2','frontend2-0']

# 2a) Pre-aggregate anomaly counts by exact minute_ts and component for quick lookup
anoms_by_min_comp = (
    anomalies_df
    .groupby(['ts', 'cmdb_id'], sort=False)
    .size()
    .rename('anomaly_count_at_minute')
    .reset_index()
    .rename(columns={'ts': 'minute_ts'})
)

# 2b/2c) Earliest anomaly in window per component and the associated kpi (first by time)
earliest_per_comp = (
    anomalies_df
    .sort_values(['ts'])
    .groupby('cmdb_id', sort=False)
    .agg(earliest_anomaly_in_window=('ts', 'min'),
         first_anomalous_kpi=('kpi_name', 'first'))
    .reset_index()
)

# make lookup dictionaries
anoms_lookup = anoms_by_min_comp.set_index(['minute_ts','cmdb_id'])['anomaly_count_at_minute'].to_dict()
earliest_lookup = earliest_per_comp.set_index('cmdb_id')['earliest_anomaly_in_window'].to_dict()
first_kpi_lookup = earliest_per_comp.set_index('cmdb_id')['first_anomalous_kpi'].to_dict()

# Build rows for the two failure minutes and components list
rows = []
for minute in failure_minutes:
    # get log_count from error_logs_per_min
    log_count = int(error_logs_per_min.loc[error_logs_per_min['minute_ts'] == minute, 'log_count'].iat[0])
    for comp in components:
        ac = int(anoms_lookup.get((minute, comp), 0))
        ear = earliest_lookup.get(comp, pd.NaT)
        kpi = first_kpi_lookup.get(comp, None)
        # include only if anomaly_count_at_minute>0 OR earliest_anomaly_in_window not null
        if (ac > 0) or (pd.notna(ear)):
            rows.append({
                'minute_ts': minute,
                'component': comp,
                'log_count': log_count,
                'anomaly_count_at_minute': ac,
                'earliest_anomaly_in_window': ear,
                'first_anomalous_kpi': kpi
            })

failure_minute_component_timeline = pd.DataFrame(rows)

# Order results by minute_ts then anomaly_count desc for readability and limit to <=20 rows
failure_minute_component_timeline = failure_minute_component_timeline.sort_values(
    ['minute_ts', 'anomaly_count_at_minute'], ascending=[True, False]
).reset_index(drop=True).head(20)

# Display compact table
failure_minute_component_timeline

```
Out[7]:
```
```python
# Plain-English summary of the failure-minute comparison using cached failure_minute_component_timeline
from datetime import datetime
import pandas as pd

# use cached DataFrame failure_minute_component_timeline
df = failure_minute_component_timeline.copy()

# Identify the two failure minutes (unique minutes in df, in order)
failure_minutes = df['minute_ts'].sort_values().unique().tolist()

rows_by_min = {m: df[df['minute_ts'] == m] for m in failure_minutes}

lines = []
lines.append(f"Two candidate failure minutes (UTC+8): {', '.join(str(m) for m in failure_minutes)}.\n")

for m in failure_minutes:
    sub = rows_by_min[m]
    # top 3 components by anomaly_count_at_minute
    top3 = sub.sort_values('anomaly_count_at_minute', ascending=False).head(3)
    top3_list = [f"{r.component} ({int(r.anomaly_count_at_minute)})" for r in top3.itertuples()]
    lines.append(f"Minute {m}: log_count={int(sub['log_count'].iat[0])}; top components by metric anomalies: {', '.join(top3_list)}.")
    # earliest anomaly time per component (from the cached earliest_anomaly_in_window field)
    # For the listed components check earliest anomaly in window (they are present in df)
    earliest_times = sub[['component','earliest_anomaly_in_window']].drop_duplicates().sort_values('component')
    # summarize if all earliest times are the same
    unique_times = earliest_times['earliest_anomaly_in_window'].dropna().unique()
    if len(unique_times) == 0:
        lines.append("  No earliest anomaly timestamps found for these components in the window.")
    elif len(unique_times) == 1:
        lines.append(f"  Earliest metric-anomaly timestamp for these components: {unique_times[0]} (all matched).")
    else:
        # show a compact sample of earliest times
        sample = ", ".join(f"{row.component}:{row.earliest_anomaly_in_window}" for row in earliest_times.itertuples(index=False))
        lines.append(f"  Earliest anomaly times by component: {sample}.")

lines.append("\nConclusion: Metric anomalies for major components (notably adservice/adservice2 with ~300 anomaly rows and frontends with ~70–115) already existed at 2022-03-20 08:30:00+08:00 — i.e., at or before the first log spike (08:33) and earlier than the second spike (08:42). This timing suggests metric anomalies began before the observed log bursts, so metrics precede the log spikes in the incident timeline.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

minute_ts     component  log_count  anomaly_count_at_minute earliest_anomaly_in_window                                first_anomalous_kpi
0  2022-03-20 08:33:00+08:00     adservice         21                      306  2022-03-20 08:30:00+08:00                                        app.grpc.rr
1  2022-03-20 08:33:00+08:00    adservice2         21                      305  2022-03-20 08:30:00+08:00  runtime.java_lang_GarbageCollector_LastGcInfo_...
2  2022-03-20 08:33:00+08:00    frontend-0         21                      112  2022-03-20 08:30:00+08:00      container.node-5.container_fs_reads./dev/vda1
3  2022-03-20 08:33:00+08:00   frontend2-0         21                       85  2022-03-20 08:30:00+08:00  mesh.source.frontend2.unknown.istio_response_b...
4  2022-03-20 08:33:00+08:00    frontend-1         21                       83  2022-03-20 08:30:00+08:00  container.node-5.container_network_receive_pac...
5  2022-03-20 08:33:00+08:00   adservice-0         21                       75  2022-03-20 08:30:00+08:00             container.node-5.container_memory_swap
6  2022-03-20 08:33:00+08:00   adservice-1         21                       71  2022-03-20 08:30:00+08:00  mesh.source.adservice.jaeger-collector.istio_r...
7  2022-03-20 08:33:00+08:00    frontend-2         21                       68  2022-03-20 08:30:00+08:00      container.node-5.container_start_time_seconds
8  2022-03-20 08:33:00+08:00  adservice2-0         21                       64  2022-03-20 08:30:00+08:00  container.node-6.container_cpu_cfs_throttled_p...
9  2022-03-20 08:33:00+08:00   adservice-2         21                       57  2022-03-20 08:30:00+08:00  container.node-5.container_cpu_cfs_throttled_p...
10 2022-03-20 08:42:00+08:00    adservice2         21                      301  2022-03-20 08:30:00+08:00  runtime.java_lang_GarbageCollector_LastGcInfo_...
11 2022-03-20 08:42:00+08:00     adservice         21                      297  2022-03-20 08:30:00+08:00                                        app.grpc.rr
12 2022-03-20 08:42:00+08:00    frontend-1         21                      115  2022-03-20 08:30:00+08:00  container.node-5.container_network_receive_pac...
13 2022-03-20 08:42:00+08:00   frontend2-0         21                      100  2022-03-20 08:30:00+08:00  mesh.source.frontend2.unknown.istio_response_b...
14 2022-03-20 08:42:00+08:00    frontend-2         21                       85  2022-03-20 08:30:00+08:00      container.node-5.container_start_time_seconds
15 2022-03-20 08:42:00+08:00    frontend-0         21                       83  2022-03-20 08:30:00+08:00      container.node-5.container_fs_reads./dev/vda1
16 2022-03-20 08:42:00+08:00   adservice-1         21                       67  2022-03-20 08:30:00+08:00  mesh.source.adservice.jaeger-collector.istio_r...
17 2022-03-20 08:42:00+08:00   adservice-0         21                       65  2022-03-20 08:30:00+08:00             container.node-5.container_memory_swap
18 2022-03-20 08:42:00+08:00  adservice2-0         21                       64  2022-03-20 08:30:00+08:00  container.node-6.container_cpu_cfs_throttled_p...
19 2022-03-20 08:42:00+08:00   adservice-2         21                       58  2022-03-20 08:30:00+08:00  container.node-5.container_cpu_cfs_throttled_p...```
```